# Phase 4 — GGUF Export

**Goal:** convert the best PyTorch checkpoint from each method to GGUF so they can run on Android, iOS, and Raspberry Pi via llama.cpp.

**GGUF mapping:**

| Bits | GGUF type | Notes |
|---|---|---|
| 4 | `Q4_K_M` | best-quality 4-bit, mixed k-quants |
| 8 | `Q8_0` | near-lossless 8-bit |
| 16 | `F16` | half precision, used as F32 baseline export |

**Inputs:** `models/checkpoints/<method>_int{4,8}/...` from notebooks 02–05 (mounted as Kaggle datasets).

**Outputs:** `models/gguf/<experiment>_Q4_K_M.gguf` etc.

**Estimated time on T4:** ~15 min per checkpoint (model load + HF dump + GGUF convert + quantize).

## 1. Setup — clone repo, install package, fetch llama.cpp

GGUF export needs two llama.cpp tools:
1. `convert_hf_to_gguf.py` — Python script that reads HuggingFace format and writes F16 GGUF.
2. `llama-quantize` binary — built from llama.cpp source; quantizes F16 GGUF to Q4_K_M / Q8_0.

We clone llama.cpp and build it once. On a T4 the build takes ~3 min.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

WORKING_DIR  = "/kaggle/working"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
REPO_DATASET = "/kaggle/input/sqat-repo"
LLAMA_CPP    = f"{WORKING_DIR}/llama.cpp"

if not os.path.exists(REPO_DIR):
    if os.path.exists(REPO_DATASET):
        shutil.copytree(REPO_DATASET, REPO_DIR)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install -q -e ".[viz]" peft
# llama.cpp's convert script needs these
!pip install -q sentencepiece protobuf gguf

In [ ]:
if not os.path.exists(LLAMA_CPP):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", LLAMA_CPP],
        check=True,
    )

# Build llama-quantize (CPU-only is fine; we're not running inference here).
build_dir = f"{LLAMA_CPP}/build"
if not os.path.exists(f"{build_dir}/bin/llama-quantize"):
    subprocess.run(["cmake", "-B", build_dir, "-S", LLAMA_CPP, "-DGGML_CUDA=OFF"], check=True)
    subprocess.run(["cmake", "--build", build_dir, "--target", "llama-quantize", "-j", "4"], check=True)

print("llama-quantize:", f"{build_dir}/bin/llama-quantize")
!ls {build_dir}/bin/ 2>/dev/null | head

## 2. Locate checkpoints

Mount the previous notebooks' output datasets so the export can find each method's `final.pt` / `final_adapter/`. Edit `CHECKPOINT_ROOTS` to match the dataset names you used.

In [ ]:
# Each entry: (experiment_name, method, target_bits, checkpoint_path, config_path)
#   - QAT / scheduled / standard:  .pt file
#   - LoRA:                        adapter directory
#   - PTQ:                         no checkpoint persisted by trainer.run_experiment;
#                                  llama.cpp's own Q4_K_M / Q8_0 conversion of the
#                                  FP32 base is the practical equivalent. We list
#                                  the FP32 base under "ptq_*" so the export still
#                                  produces matching GGUF files for the comparison.
#
# Adjust these dataset roots to match what you uploaded.
CHECKPOINT_ROOTS = {
    "ptq":           None,                                # FP32 base from cache
    "standard_qat":  "/kaggle/input/sqat-standard-qat",
    "scheduled_qat": "/kaggle/input/sqat-scheduled-qat",
    "lora_qat":      "/kaggle/input/sqat-lora-qat",
}
FP32_BASE = f"{WORKING_DIR}/models/base"

EXPORT_TARGETS = []

def add(name, method, bits, ckpt_subpath, cfg_path):
    if method == "ptq":
        # Re-quantize the FP32 base from HF Hub via llama.cpp.
        EXPORT_TARGETS.append((name, method, bits, "HF_HUB", cfg_path))
        return
    root = Path(CHECKPOINT_ROOTS[method])
    ckpt = root / ckpt_subpath
    if ckpt.exists():
        EXPORT_TARGETS.append((name, method, bits, ckpt, cfg_path))
    else:
        print(f"  SKIP — checkpoint not found: {ckpt}")

add("ptq_int4",                  "ptq",           4, None,                                                    "configs/ptq/ptq_int4.yaml")
add("ptq_int8",                  "ptq",           8, None,                                                    "configs/ptq/ptq_int8.yaml")
add("standard_qat_int4",         "standard_qat",  4, "models/checkpoints/standard_qat_int4/final.pt",         "configs/standard_qat/qat_int4.yaml")
add("standard_qat_int8",         "standard_qat",  8, "models/checkpoints/standard_qat_int8/final.pt",         "configs/standard_qat/qat_int8.yaml")
add("scheduled_qat_cosine_int4", "scheduled_qat", 4, "models/checkpoints/scheduled_qat_cosine_int4/final.pt", "configs/scheduled_qat/scheduled_cosine_int4.yaml")
add("lora_qat_int4",             "lora_qat",      4, "models/checkpoints/lora_qat_int4/final_adapter",        "configs/lora_qat/lora_qat_int4.yaml")
add("lora_qat_int8",             "lora_qat",      8, "models/checkpoints/lora_qat_int8/final_adapter",        "configs/lora_qat/lora_qat_int8.yaml")

for t in EXPORT_TARGETS:
    print(f"  {t[0]:30s}  {t[3]}")
print(f"\n{len(EXPORT_TARGETS)} checkpoints ready to export.")

## 3. Export each checkpoint to GGUF

In [ ]:
import gc
import torch
from src.utils.config_loader import load_config
from src.utils.export import export_from_config, export_hf_model

GGUF_OUT = Path(WORKING_DIR) / "models" / "gguf"
GGUF_OUT.mkdir(parents=True, exist_ok=True)

exports = []
for name, method, bits, ckpt, cfg_path in EXPORT_TARGETS:
    print(f"\n=== Exporting {name} ===")
    cfg = load_config(cfg_path)
    cfg.model.cache_dir = FP32_BASE          # match where notebooks 01-05 cached weights
    try:
        if method == "ptq":
            # No saved checkpoint: quantize the FP32 base via llama.cpp's own k-quants.
            result = export_hf_model(
                model_name=cfg.model.name,
                cache_dir=cfg.model.cache_dir,
                output_dir=GGUF_OUT,
                target_bits=bits,
                output_basename=name,
                llama_cpp_dir=LLAMA_CPP,
                keep_f16_gguf=False,
            )
        else:
            result = export_from_config(
                cfg,
                checkpoint_path=ckpt,
                output_dir=GGUF_OUT,
                llama_cpp_dir=LLAMA_CPP,
                device_str="cuda" if torch.cuda.is_available() else "cpu",
                keep_f16_gguf=False,
            )
        print(result.summary())
        exports.append((name, result))
    except Exception as e:
        print(f"FAILED — {name}: {type(e).__name__}: {e}")
    gc.collect(); torch.cuda.empty_cache()

print(f"\nSuccessfully exported {len(exports)}/{len(EXPORT_TARGETS)} checkpoints.")

## 4. Verify each GGUF runs

Smoke test: load each GGUF with `llama_cpp.Llama` and generate a few tokens. Catches conversion errors that don't show up at export time.

In [ ]:
!pip install -q llama-cpp-python

from llama_cpp import Llama

SAMPLE_PROMPTS = [
    "The capital of France is",
    "Python is a programming language used for",
    "Once upon a time, in a small village,",
    "The chemical symbol for gold is",
    "To improve your sleep quality, you should",
]
MAX_NEW_TOKENS = 30

gguf_outs = {}    # {experiment_name: [completion0, completion1, ...]}
for name, result in exports:
    try:
        llm = Llama(model_path=str(result.gguf_path), n_ctx=512, n_threads=4, verbose=False)
        completions = []
        for p in SAMPLE_PROMPTS:
            out = llm(p, max_tokens=MAX_NEW_TOKENS, echo=False, temperature=0.0)
            completions.append(out["choices"][0]["text"].strip())
        gguf_outs[name] = completions
        print(f"  OK  {name:30s}  ({result.file_size_gb:.2f} GB)")
        del llm
    except Exception as e:
        print(f"  FAIL {name:30s}  {type(e).__name__}: {e}")
        gguf_outs[name] = ["[error]"] * len(SAMPLE_PROMPTS)

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# Cross-method GGUF inference comparison — same five prompts as notebooks 02-05.
inference_df = pd.DataFrame({"Prompt": SAMPLE_PROMPTS, **gguf_outs})
inference_df.to_json(Path(WORKING_DIR) / "results" / "gguf_inference.json",
                     orient="records", indent=2)
display(HTML(inference_df.to_html(index=False, escape=False).replace(
    "<table", '<table style="font-family:monospace; font-size:11px"')))

## 5. File-size summary

In [ ]:
import plotly.graph_objects as go
import pandas as pd

rows = [{
    "experiment":  name,
    "gguf_type":   r.gguf_type,
    "size_gb":     round(r.file_size_gb, 3),
    "export_time": round(r.export_time_seconds, 1),
    "path":        r.gguf_path.name,
} for name, r in exports]
size_df = pd.DataFrame(rows).sort_values("size_gb")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=size_df["experiment"], y=size_df["size_gb"],
    marker_color=["#4C72B0" if t.startswith("Q8") or t == "F16" else "#C44E52"
                  for t in size_df["gguf_type"]],
    text=[f"{s:.2f} GB" for s in size_df["size_gb"]],
    textposition="outside",
    customdata=size_df[["gguf_type", "export_time"]],
    hovertemplate="<b>%{x}</b><br>size=%{y:.2f} GB<br>type=%{customdata[0]}<br>export=%{customdata[1]}s",
))
fig.add_hline(y=6.5, line_dash="dash", line_color="black", annotation_text="FP32 (6.5 GB)")
fig.update_layout(title="GGUF file sizes across methods",
                  xaxis_title="experiment", yaxis_title="size (GB)",
                  height=440, showlegend=False,
                  margin=dict(t=80, b=80, l=60, r=20))
fig.show()
size_df

In [ ]:
!ls -lh {GGUF_OUT}/

## Next steps

Save the GGUF dir as a Kaggle dataset (`sqat-gguf`). The Android / iOS / RPi wrappers in `examples/` consume those files directly.

Proceed to `07_benchmarks.ipynb` for the cross-method analysis.